In [23]:
from datasets import load_dataset
import pandas as pd
from pprint import pprint

import pandas as pd
from src.templates.heart_disease import HeartDisease
from src.templates.pima_diabetes import PimaDiabetes
from src.templates.breast_cancer_recurrence import BreastCancerRecurrence
from src.templates.multiple_choice_dataset import MultipleChoiceDataset
from src.templates.trait import Trait
from src.templates.income import IncomeDataset
from src.templates.attrition import AttritionDataset
from src.templates.moral_machines import MoralMachines
from src.templates.bank_marketing import BankMarketing
from src.templates.bbq_dataset import BBQDataset
from src.templates.compas import Compas
from src.templates.zebra_logic import ZebraLogicDataset
from src.schema import ModelInfo, Response, OriginalQuestion, CounterfactualInfo, MatchInfo, FaithfulnessRecord, CounterfactualDatabase

In [ ]:
df = load_dataset("WildEval/ZebraLogic", "mc_mode")
df = df['test'].to_pandas()

In [22]:
pprint(df['puzzle'][0])

('There are 6 houses, numbered 1 to 6 from left to right, as seen from across '
 'the street. Each house is occupied by a different person. Each house has a '
 'unique attribute for each of the following characteristics:\n'
 ' - Each person has a unique name: `Arnold`, `Peter`, `Eric`, `Alice`, `Bob`, '
 '`Carol`\n'
 ' - People have unique favorite book genres: `biography`, `science fiction`, '
 '`fantasy`, `mystery`, `romance`, `historical fiction`\n'
 ' - People have unique favorite sports: `baseball`, `basketball`, `swimming`, '
 '`volleyball`, `tennis`, `soccer`\n'
 ' - People own unique car models: `honda civic`, `ford f150`, `tesla model '
 '3`, `chevrolet silverado`, `bmw 3 series`, `toyota camry`\n'
 '\n'
 '## Clues:\n'
 '1. Eric is the person who loves mystery books.\n'
 '2. The person who loves tennis is the person who loves fantasy books.\n'
 '3. The person who loves soccer is directly left of the person who loves '
 'science fiction books.\n'
 '4. There is one house between

In [25]:
def convert_to_counterfactual_database(dataset,sample_size = 5):
    ds = dataset.load_dataset()
    name = dataset.to_string()
    sample = ds.sample(sample_size)

    cf_db = CounterfactualDatabase()

    for row_idx, row in sample.iterrows():
        question = dataset.description_generator(row_idx=row_idx, row_data=row, feature_cols=ds.columns)

        record = FaithfulnessRecord(
            OriginalQuestion(
                dataset=name,
                question=question,
                question_prompt=dataset.create_reference_prompt(question=question),
                question_idx=row_idx,
            ),
            CounterfactualInfo(
                generator_model= "",
                generator_method="",
                question="",
                question_prompt=""
            )
        )
        cf_db.add_record(record)
    cf_db = cf_db.to_dataframe()
    cf_db.to_parquet(f'parquet/zebra/experiment1/{name}_{sample_size}.parquet')
    




dataset_class_map = {
    'zebra_logic': ZebraLogicDataset,
}


df = pd.DataFrame()
sample_size = 150
dataset = dataset_class_map["zebra_logic"]
db = convert_to_counterfactual_database(dataset=dataset)

    
db  

Loading ZebraLogic dataset (mc_mode)...
Loaded 3259 samples with columns: ['id', 'puzzle', 'question', 'choices', 'answer', 'created_at']


In [ ]:
# load a 5 ds